# Hay fedele: soma singolo, dataset bilanciato e ConvLSTM Large

Questo notebook ricrea un compartimento somatico isopotenziale usando direttamente le cinetiche e i parametri di `L5PCbiophys3.hoc` (Hay 2011, Figura 4). Conserva tutti i 17 stati intrinseci del soma e aggiunge AMPA, NMDA e GABA-A come tre stati di input esterni. Il dataset separa le traiettorie tra train/validation/test, bilancia cinque regimi di stimolazione, mostra avanzamento ed ETA e viene riutilizzato automaticamente se già disponibile.


In [ ]:
from pathlib import Path
import subprocess, sys, tempfile

def project_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    work = Path('/kaggle/working')
    if work.exists():
        candidates += [p.parent for p in work.glob('*/pyproject.toml')]
    for candidate in candidates:
        marker = candidate / 'pyproject.toml'
        if marker.exists() and 'hay-single-compartment' in marker.read_text(encoding='utf-8'):
            return candidate
    destination = Path(tempfile.mkdtemp(prefix='hay_faithful_', dir='/kaggle/working'))
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(destination)])
    return destination

ROOT = project_root()
SRC = ROOT / 'src'
assert (SRC / 'hay_single_compartment').is_dir(), f'Package source missing: {SRC}'
sys.path.insert(0, str(SRC))
print('Project:', ROOT)


In [ ]:
import h5py, json, os, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display
from hay_single_compartment import (
    FAITHFUL_INPUT_NAMES, FAITHFUL_STATE_NAMES, FaithfulSimulationConfig,
    generate_faithful_dataset, validate_faithful_dataset,
)
from hay_single_compartment.dataset import Normalization
from hay_single_compartment.models import build_model
from hay_single_compartment.training import rollout_batch, train_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('/kaggle/working/hay_faithful_soma_01') if Path('/kaggle').exists() else ROOT / 'artifacts' / 'faithful_soma_01'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_NAME = 'hay_faithful_soma_balanced_v1.h5'
print('Device:', DEVICE, '| output:', OUTPUT_DIR)
print('State dimension:', len(FAITHFUL_STATE_NAMES), '| input dimension:', len(FAITHFUL_INPUT_NAMES))


## 1. Trova la cache oppure genera il dataset

L'ordine è: dataset Kaggle montato in `/kaggle/input`, cache della sessione in `/kaggle/working`, nuova generazione. Il file montato è read-only ma può essere usato direttamente per il training. Per conservarlo tra sessioni, dopo la prima esecuzione usare **Save Version** e pubblicare `hay_faithful_soma_balanced_v1.h5` come Kaggle Dataset; nelle esecuzioni successive basta aggiungerlo agli input del notebook.


In [ ]:
config = FaithfulSimulationConfig(
    duration_ms=2000.0, warmup_ms=500.0, seed=27182,
    train_trajectories=24, validation_trajectories=4, test_trajectories=6,
)
working_dataset = OUTPUT_DIR / DATASET_NAME
mounted = list(Path('/kaggle/input').rglob(DATASET_NAME)) if Path('/kaggle/input').exists() else []
if mounted:
    DATASET = mounted[0]
    dataset_report = validate_faithful_dataset(DATASET)
    with h5py.File(DATASET, 'r') as h5:
        config = FaithfulSimulationConfig.from_dict(json.loads(h5.attrs['config_json']))
    dataset_report['cache_hit'] = True
    dataset_report['cache_source'] = 'kaggle_input'
    print('[dataset] mounted cache:', DATASET)
else:
    DATASET = working_dataset
    dataset_report = generate_faithful_dataset(DATASET, config, progress=True, reuse=True, workers=min(4, os.cpu_count() or 1))
    dataset_report['cache_source'] = 'working_cache' if dataset_report['cache_hit'] else 'generated'
print(json.dumps(dataset_report, indent=2))


In [ ]:
rows = []
for split, summary in dataset_report['splits'].items():
    for regime, fraction in summary['regime_fractions'].items():
        rows.append({'split': split, 'regime': regime, 'fraction': fraction})
balance_table = pd.DataFrame(rows)
display(balance_table.pivot(index='regime', columns='split', values='fraction'))
ax = balance_table.pivot(index='regime', columns='split', values='fraction').plot.bar(figsize=(11, 4))
ax.axhline(0.2, color='black', ls='--', lw=1, label='ideal 20%')
ax.set_ylabel('fraction of samples'); ax.set_title('Regime coverage by isolated split'); ax.legend()
plt.tight_layout(); plt.show()


## 2. Ispezione del teacher

Il dataset salva tutti i 20 stati Markoviani, le 14 correnti più il totale, input, spike, regime, seed e configurazione. In particolare include `h_Nap_Et2`, `h_K_Pst`, `h_Ca_HVA` e il calcio somatico con le costanti lente originali.


In [ ]:
with h5py.File(DATASET, 'r') as h5:
    states = h5['train/states'][0]
    inputs = h5['train/inputs'][0]
    regimes = h5['train/regimes'][0]
time_ms = np.arange(len(states)) * config.dt_ms
state_index = {name: i for i, name in enumerate(FAITHFUL_STATE_NAMES)}
fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)
axes[0].plot(time_ms, states[:, state_index['v_mV']], lw=.8); axes[0].set_ylabel('V (mV)')
axes[1].plot(time_ms, 1000 * states[:, state_index['ca_i_mM']], lw=.9); axes[1].set_ylabel('Ca_i (uM)')
axes[2].plot(time_ms, states[:, state_index['h_Nap_Et2']], label='h_Nap')
axes[2].plot(time_ms, states[:, state_index['h_K_Pst']], label='h_KP')
axes[2].plot(time_ms, states[:, state_index['h_Ca_HVA']], label='h_CaHVA'); axes[2].legend(); axes[2].set_ylabel('slow gates')
axes[3].plot(time_ms[:-1], inputs[:, 0], lw=.8, label='I injected')
axes[3].step(time_ms[:-1], regimes, where='post', alpha=.5, label='regime'); axes[3].set_ylabel('drive'); axes[3].set_xlabel('time (ms)'); axes[3].legend()
plt.tight_layout(); plt.show()


## 3. Nuovo baseline controllato: solo ConvLSTM Large

Non trasferiamo i vecchi pesi: lo schema è passato da 17 a 20 stati e la funzione teacher è diversa. Ripartiamo dalla ConvLSTM Large, l'architettura generale già validata, e salviamo il miglior checkpoint per validation loss. Training, epoche e rollout mostrano avanzamento ed ETA.


In [ ]:
settings = dict(hidden_dim=128, layers=3, width_multiplier=2)
probe = build_model('conv_lstm', len(FAITHFUL_STATE_NAMES) + len(FAITHFUL_INPUT_NAMES), len(FAITHFUL_STATE_NAMES), **settings)
print('ConvLSTM Large parameters:', f'{sum(p.numel() for p in probe.parameters()):,}')
report = train_model(
    DATASET, OUTPUT_DIR / 'models', 'conv_lstm', run_name='faithful_conv_large',
    epochs=50, sequence_length=256, stride=64, batch_size=24,
    learning_rate=5e-4, dropout=0.1, patience=10, minimum_epochs=25,
    device=DEVICE, seed=27182, use_amp=True, verbose=True, **settings,
)
pd.DataFrame([{'run': report['run_name'], 'parameters': report['parameters'], 'epochs': report['epochs_trained'], 'validation_loss': report['best_validation_loss'], 'test_voltage_rmse_mV': report['test']['voltage_rmse_mv'], 'test_normalized_rmse': report['test']['mean_normalized_rmse']}])


In [ ]:
history = pd.DataFrame(report['history'])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.epoch, history.train_loss, label='train'); axes[0].plot(history.epoch, history.validation_loss, label='validation'); axes[0].set_yscale('log'); axes[0].legend(); axes[0].set_title('One-step objective')
per_state = pd.Series(report['test']['per_state_rmse'])
(per_state / Normalization.from_h5(DATASET).state_std).sort_values().plot.barh(ax=axes[1]); axes[1].set_title('Normalized RMSE by state')
plt.tight_layout(); plt.show()


## 4. Rollout completamente autoregressivo

Dopo lo stato iniziale, tutti i 20 stati predetti vengono reinseriti. Rimangono teacher soltanto i quattro input esterni. Le finestre arrivano a 2 s perché il nuovo sistema contiene dinamiche di centinaia di millisecondi e `h_Nap` ha tau di circa 2 s.


In [ ]:
checkpoint = torch.load(report['checkpoint'], map_location=DEVICE, weights_only=False)
model = build_model(checkpoint['architecture'], 24, 20, **checkpoint['model_kwargs']).to(DEVICE)
model.load_state_dict(checkpoint['model_state'])
normalization = Normalization.from_dict(checkpoint['normalization'])
with h5py.File(DATASET, 'r') as h5:
    truth = h5['test/states'][...]
    future_inputs = h5['test/inputs'][...]
prediction = rollout_batch(model, truth[:, 0], future_inputs, normalization, DEVICE, progress=True)
rows = []
for horizon in (50, 100, 200, 500, 1000, 1500, 2000):
    end = min(int(horizon / config.dt_ms) + 1, truth.shape[1])
    error = prediction[:, :end] - truth[:, :end]
    rows.append({'horizon_ms': horizon, 'voltage_rmse_mV': float(np.sqrt(np.mean(error[..., 0]**2))), 'mean_normalized_rmse': float(np.sqrt(np.mean((error / normalization.state_std)**2, axis=(0, 1))).mean())})
rollout_table = pd.DataFrame(rows)
rollout_table.to_csv(OUTPUT_DIR / 'rollout_horizons.csv', index=False)
rollout_table


In [ ]:
t = np.arange(truth.shape[1]) * config.dt_ms
fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)
axes[0].plot(t, truth[0, :, 0], color='black', lw=1, label='teacher'); axes[0].plot(t, prediction[0, :, 0], lw=.9, label='conv_large'); axes[0].set_ylabel('V (mV)'); axes[0].legend()
for name in ('h_Nap_Et2', 'h_K_Pst', 'h_Ca_HVA'):
    i = state_index[name]; axes[1].plot(t, truth[0, :, i], lw=1, label=f'{name} teacher'); axes[1].plot(t, prediction[0, :, i], ls='--', lw=.9, label=f'{name} model')
axes[1].set_ylabel('slow gates'); axes[1].legend(ncol=3)
axes[2].plot(t, 1000 * truth[0, :, 1], color='black', label='Ca teacher'); axes[2].plot(t, 1000 * prediction[0, :, 1], label='Ca model'); axes[2].set_ylabel('Ca_i (uM)'); axes[2].set_xlabel('time (ms)'); axes[2].legend()
plt.tight_layout(); fig.savefig(OUTPUT_DIR / 'faithful_rollout.png', dpi=160); plt.show()


## 5. Salvataggio persistente

Il link seguente è utile nella sessione corrente. Il metodo più affidabile su Kaggle è **Save Version → Save & Run All**, quindi scaricare il file dalla sezione Output oppure creare un Kaggle Dataset dagli output. Non viene trasformato in Base64, evitando blocchi del browser sui file grandi.


In [ ]:
export_dataset = Path('/kaggle/working') / DATASET_NAME if Path('/kaggle').exists() else OUTPUT_DIR / DATASET_NAME
if DATASET.resolve() != export_dataset.resolve():
    shutil.copy2(DATASET, export_dataset)
manifest_source = DATASET.with_suffix('.manifest.json')
if manifest_source.exists():
    shutil.copy2(manifest_source, export_dataset.with_suffix('.manifest.json'))
print('Dataset persistente:', export_dataset, f'({export_dataset.stat().st_size / 2**20:.1f} MiB)')
display(FileLink(str(export_dataset)))

results_stage = Path('/kaggle/working/hay_faithful_results_only') if Path('/kaggle').exists() else OUTPUT_DIR.parent / 'hay_faithful_results_only'
if results_stage.exists(): shutil.rmtree(results_stage)
shutil.copytree(OUTPUT_DIR, results_stage, ignore=lambda path, names: {name for name in names if name.endswith(('.h5', '.pt', '.partial'))})
results_zip = shutil.make_archive('/kaggle/working/hay_faithful_soma_01_results' if Path('/kaggle').exists() else str(OUTPUT_DIR / 'hay_faithful_soma_01_results'), 'zip', root_dir=results_stage.parent, base_dir=results_stage.name)
print('Risultati:', results_zip)
display(FileLink(results_zip))
